# 🩺 Fetal Image Analysis Using Deep Learning
## PHASE 2 — MODEL TRAINING PIPELINE
### TB Solutions | ECE Microproject

---

**Project:** Automated fetal ultrasound image analysis using YOLOv8  
**Dataset:** BCNatal Maternal-Fetal Ultrasound Dataset (~12,400 images, 9 classes)  
**Model:** YOLOv8s-cls (Ultralytics) for classification  
**Phase:** 2 of 5 — Model Training  

---

**Prerequisites:**  
- Phase 1 notebook has been run successfully  
- Dataset is prepared in `/content/drive/MyDrive/FetalNet/`  
- `data.yaml` exists on Google Drive  

**How to use this notebook:**
1. Open in Google Colab
2. Go to **Runtime → Change runtime type → Select T4 GPU**
3. Run each cell **top to bottom** (Shift + Enter)
4. Training takes ~20-40 minutes on a T4 GPU
5. All outputs are saved to Google Drive automatically

---

# ═══════════════════════════════════════════
# STEP 1 — ENVIRONMENT & DRIVE SETUP
# ═══════════════════════════════════════════

**What this step does:**  
Mounts Google Drive, verifies Phase 1 outputs exist, checks GPU availability,  
and installs/updates the Ultralytics library.

**Expected output:**  
Confirmation that Drive is mounted, dataset folders exist, GPU info printed.

In [ ]:
# ============================================================
# STEP 1.1 — Mount Google Drive
# ============================================================
# We mount Drive so we can access Phase 1 outputs and save
# all training results persistently.

from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")

In [ ]:
# ============================================================
# STEP 1.2 — Install / upgrade required libraries
# ============================================================

!pip install -q --upgrade ultralytics
!pip install -q torch torchvision
!pip install -q opencv-python-headless
!pip install -q matplotlib seaborn
!pip install -q numpy pandas
!pip install -q Pillow
!pip install -q scikit-learn
!pip install -q pyyaml

print("\n" + "="*60)
print("✅ All libraries installed / updated!")
print("="*60)

In [ ]:
# ============================================================
# STEP 1.3 — Import all libraries and print versions
# ============================================================

import os
import sys
import time
import shutil
import glob
import random
import warnings
import json
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import cv2
import torch
import torchvision
import yaml
from PIL import Image
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    top_k_accuracy_score
)
import ultralytics
from ultralytics import YOLO

warnings.filterwarnings('ignore')

# ---- Print versions ----
print("=" * 60)
print("📦 LIBRARY VERSIONS")
print("=" * 60)

libraries = {
    "Python":        sys.version.split()[0],
    "NumPy":         np.__version__,
    "Pandas":        pd.__version__,
    "Matplotlib":    plt.matplotlib.__version__,
    "Seaborn":       sns.__version__,
    "OpenCV":        cv2.__version__,
    "PyTorch":       torch.__version__,
    "TorchVision":   torchvision.__version__,
    "Pillow":        Image.__version__,
    "Scikit-learn":  __import__('sklearn').__version__,
    "Ultralytics":   ultralytics.__version__,
}

for lib, ver in libraries.items():
    print(f"  {lib:<16} : {ver}")

print("=" * 60)

In [ ]:
# ============================================================
# STEP 1.4 — Check GPU availability (nvidia-smi)
# ============================================================
# We check for a GPU and configure the device accordingly.
# If no GPU is found, we fall back to CPU mode automatically.

print("=" * 60)
print("🖥️  GPU STATUS")
print("=" * 60)

# Run nvidia-smi to show GPU details
!nvidia-smi

print("\n" + "-" * 60)

# Programmatic GPU check
GPU_AVAILABLE = torch.cuda.is_available()

if GPU_AVAILABLE:
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    CUDA_VERSION = torch.version.cuda
    DEVICE = 0  # GPU index

    print(f"🟢 GPU Detected!")
    print(f"   Name         : {GPU_NAME}")
    print(f"   VRAM         : {GPU_VRAM_GB:.1f} GB")
    print(f"   CUDA Version : {CUDA_VERSION}")
    print(f"   Device       : cuda:0")
else:
    GPU_NAME = "None"
    GPU_VRAM_GB = 0
    CUDA_VERSION = "N/A"
    DEVICE = 'cpu'

    print("🔴 NO GPU DETECTED!")
    print("   ⚠️  Training will run on CPU (VERY SLOW).")
    print("   💡 Go to Runtime → Change runtime type → T4 GPU")
    print("   Falling back to CPU mode automatically...")

print("=" * 60)

In [ ]:
# ============================================================
# STEP 1.5 — Set global paths and verify Phase 1 outputs
# ============================================================
# All configurable paths are defined here. We verify that every
# required file/folder from Phase 1 exists on Google Drive.

# Random seed for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if GPU_AVAILABLE:
    torch.cuda.manual_seed_all(SEED)

# ---- Directory paths ----
DRIVE_DIR    = "/content/drive/MyDrive/FetalNet"
DATASET_DIR  = os.path.join(DRIVE_DIR, "dataset")
YAML_PATH    = os.path.join(DRIVE_DIR, "data.yaml")
RUNS_DIR     = os.path.join(DRIVE_DIR, "runs")
RUN_NAME     = "fetal_yolov8s_v1"

# ---- Fallback: check if dataset is in /content/dataset (same session) ----
if not os.path.exists(DATASET_DIR):
    if os.path.exists("/content/dataset"):
        print("📁 Dataset found in /content/dataset (same Colab session as Phase 1)")
        DATASET_DIR = "/content/dataset"
        YAML_PATH = os.path.join(DATASET_DIR, "data.yaml")
    else:
        print("⚠️ Dataset not found! Run Phase 1 first or upload dataset to Drive.")

# ---- Verify Phase 1 outputs ----
print("=" * 60)
print("🔍 VERIFYING PHASE 1 OUTPUTS")
print("=" * 60)

required_items = {
    "data.yaml":       YAML_PATH,
    "train/ folder":   os.path.join(DATASET_DIR, "train"),
    "val/ folder":     os.path.join(DATASET_DIR, "val"),
    "test/ folder":    os.path.join(DATASET_DIR, "test"),
}

all_found = True
for name, path in required_items.items():
    exists = os.path.exists(path)
    icon = "✅" if exists else "❌"
    print(f"  {icon} {name:<20} → {path}")
    if not exists:
        all_found = False

print()
if all_found:
    print("🎉 All Phase 1 outputs verified! Ready to proceed.")
else:
    print("⚠️ Some Phase 1 outputs are missing!")
    print("   Please run Phase 1 notebook first, or upload files to Google Drive.")
    print(f"   Expected location: {DRIVE_DIR}")

print("=" * 60)

# ═══════════════════════════════════════════
# STEP 2 — MODEL SELECTION & CONFIGURATION
# ═══════════════════════════════════════════

**What this step does:**  
Auto-detects the task type (classification vs detection) from the data.yaml,  
selects the appropriate YOLOv8 model, and configures all training hyperparameters.

**Expected output:**  
A formatted table showing the full training configuration before training starts.

In [ ]:
# ============================================================
# STEP 2.1 — Load data.yaml and auto-detect task type
# ============================================================
# The BCNatal dataset is a classification dataset (no bounding
# boxes). We verify this by inspecting the data.yaml and the
# folder structure.

print("=" * 60)
print("📄 LOADING data.yaml")
print("=" * 60)

with open(YAML_PATH, 'r') as f:
    data_config = yaml.safe_load(f)

print(f"  path  : {data_config.get('path', 'N/A')}")
print(f"  train : {data_config.get('train', 'N/A')}")
print(f"  val   : {data_config.get('val', 'N/A')}")
print(f"  test  : {data_config.get('test', 'N/A')}")
print(f"  nc    : {data_config.get('nc', 'N/A')}")
print(f"  names : {data_config.get('names', 'N/A')}")

NUM_CLASSES = data_config['nc']
CLASS_NAMES = data_config['names']

# ---- Auto-detect task type ----
# Check if dataset has YOLO detection labels (.txt files with bboxes)
# or is organized as classification (images in class-named folders)
train_path = os.path.join(DATASET_DIR, data_config['train'])
has_label_files = False

# Check for .txt label files (detection format)
label_dirs = [
    os.path.join(DATASET_DIR, 'labels', 'train'),
    os.path.join(DATASET_DIR, 'train', 'labels'),
]
for ld in label_dirs:
    if os.path.exists(ld):
        txt_files = glob.glob(os.path.join(ld, '*.txt'))
        if txt_files:
            has_label_files = True
            break

# Check if train/ contains class-named subdirectories (classification format)
train_subdirs = [d for d in os.listdir(train_path)
                 if os.path.isdir(os.path.join(train_path, d))]

if has_label_files:
    TASK_TYPE = "detect"
    MODEL_VARIANT = "yolov8s.pt"
    TASK_REASON = "Bounding box label files (.txt) found in labels/ directory"
elif len(train_subdirs) > 1:
    TASK_TYPE = "classify"
    MODEL_VARIANT = "yolov8s-cls.pt"
    TASK_REASON = "Images organized in class-named subfolders (no bbox labels)"
else:
    TASK_TYPE = "classify"
    MODEL_VARIANT = "yolov8s-cls.pt"
    TASK_REASON = "Default: BCNatal is a classification dataset"

print(f"\n{'─' * 60}")
print(f"  Task Type     : {TASK_TYPE.upper()}")
print(f"  Model         : {MODEL_VARIANT}")
print(f"  Reason        : {TASK_REASON}")
print(f"  Subdirs found : {train_subdirs}")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 2.2 — Define full training configuration
# ============================================================
# All hyperparameters are defined here. We auto-adjust batch
# size based on available GPU VRAM.

# Auto-adjust batch size based on VRAM
if GPU_AVAILABLE and GPU_VRAM_GB < 8:
    BATCH_SIZE = 8
    print(f"⚠️ Low VRAM ({GPU_VRAM_GB:.1f} GB) — reducing batch size to 8")
else:
    BATCH_SIZE = 16

# ---- Training hyperparameters ----
TRAIN_CONFIG = {
    'data':        DATASET_DIR,        # Path to dataset root
    'epochs':      50,                 # Total training epochs
    'imgsz':       224,                # Input image size
    'batch':       BATCH_SIZE,         # Batch size
    'optimizer':   'AdamW',            # Optimizer
    'lr0':         0.001,              # Initial learning rate
    'lrf':         0.01,               # Final LR = lr0 * lrf
    'patience':    10,                 # Early stopping patience
    'dropout':     0.2,                # Dropout rate
    'seed':        SEED,               # Random seed
    'device':      DEVICE,             # GPU (0) or CPU
    'project':     RUNS_DIR,           # Save directory root
    'name':        RUN_NAME,           # Run name
    'save':        True,               # Save checkpoints
    'save_period':  10,                # Save every N epochs
    'val':         True,               # Run validation
    'plots':       True,               # Generate plots
    'exist_ok':    True,               # Overwrite existing run
    'verbose':     True,               # Verbose output
    'workers':     2,                  # Data loader workers
    'pretrained':  True,               # Use pretrained weights
}

# ---- Print config as formatted table ----
print("\n" + "=" * 60)
print("⚙️  TRAINING CONFIGURATION")
print("=" * 60)
print(f"  {'Parameter':<18} {'Value'}")
print(f"  {'─'*18} {'─'*38}")

for key, val in TRAIN_CONFIG.items():
    print(f"  {key:<18} {val}")

print(f"\n  {'Model variant':<18} {MODEL_VARIANT}")
print(f"  {'Task type':<18} {TASK_TYPE}")
print(f"  {'Num classes':<18} {NUM_CLASSES}")
print(f"  {'Class names':<18} {CLASS_NAMES}")
print(f"  {'GPU':<18} {GPU_NAME}")
print(f"  {'VRAM':<18} {GPU_VRAM_GB:.1f} GB")
print("=" * 60)

# ═══════════════════════════════════════════
# STEP 3 — PRE-TRAINING CHECKS
# ═══════════════════════════════════════════

**What this step does:**  
Runs final verification before training: validates data.yaml, counts images,  
checks class balance, displays a sample batch, and estimates training time.

**Expected output:**  
Image counts per split, class balance warnings, sample batch grid, and GO confirmation.

In [ ]:
# ============================================================
# STEP 3.1 — Count images in train/val/test
# ============================================================

IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'}

def count_images_recursive(directory):
    """Count all image files recursively in a directory."""
    count = 0
    if not os.path.exists(directory):
        return 0
    for root, _, files in os.walk(directory):
        for f in files:
            if os.path.splitext(f)[1].lower() in IMG_EXTENSIONS:
                count += 1
    return count

def count_per_class(split_dir):
    """Count images per class subfolder."""
    counts = {}
    if not os.path.exists(split_dir):
        return counts
    for cls_folder in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls_folder)
        if os.path.isdir(cls_path):
            counts[cls_folder] = count_images_recursive(cls_path)
    return counts


train_total = count_images_recursive(os.path.join(DATASET_DIR, 'train'))
val_total   = count_images_recursive(os.path.join(DATASET_DIR, 'val'))
test_total  = count_images_recursive(os.path.join(DATASET_DIR, 'test'))
grand_total = train_total + val_total + test_total

train_per_class = count_per_class(os.path.join(DATASET_DIR, 'train'))
val_per_class   = count_per_class(os.path.join(DATASET_DIR, 'val'))
test_per_class  = count_per_class(os.path.join(DATASET_DIR, 'test'))

print("=" * 70)
print("📊 DATASET IMAGE COUNTS")
print("=" * 70)
print(f"  {'Class':<25} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>7}")
print(f"  {'─'*25} {'─'*7} {'─'*7} {'─'*7} {'─'*7}")

all_classes = sorted(set(list(train_per_class.keys()) +
                        list(val_per_class.keys()) +
                        list(test_per_class.keys())))

low_count_classes = []
for cls in all_classes:
    t = train_per_class.get(cls, 0)
    v = val_per_class.get(cls, 0)
    te = test_per_class.get(cls, 0)
    total = t + v + te
    flag = " ⚠️" if total < 50 else ""
    if total < 50:
        low_count_classes.append(cls)
    print(f"  {cls:<25} {t:>7} {v:>7} {te:>7} {total:>7}{flag}")

print(f"  {'─'*25} {'─'*7} {'─'*7} {'─'*7} {'─'*7}")
print(f"  {'TOTAL':<25} {train_total:>7} {val_total:>7} {test_total:>7} {grand_total:>7}")
print(f"  {'PERCENTAGE':<25} {train_total/grand_total*100:>6.1f}% {val_total/grand_total*100:>6.1f}% {test_total/grand_total*100:>6.1f}%")
print("=" * 70)

# Warn about low-count classes
if low_count_classes:
    print(f"\n⚠️  LOW-COUNT WARNING: The following classes have <50 total images:")
    for cls in low_count_classes:
        total = train_per_class.get(cls,0) + val_per_class.get(cls,0) + test_per_class.get(cls,0)
        print(f"     • {cls}: {total} images")
    print("   This may cause poor performance for these classes.")
else:
    print("\n✅ All classes have 50+ images. Class balance looks good.")

In [ ]:
# ============================================================
# STEP 3.2 — Display a sample batch of 16 training images
# ============================================================
# We manually load 16 random images from the training set to
# give a visual preview of what the model will train on.

print("Loading sample batch of 16 training images...")

# Collect all training image paths with their class labels
sample_images = []
train_root = os.path.join(DATASET_DIR, 'train')

for cls_folder in sorted(os.listdir(train_root)):
    cls_path = os.path.join(train_root, cls_folder)
    if not os.path.isdir(cls_path):
        continue
    for img_file in os.listdir(cls_path):
        if os.path.splitext(img_file)[1].lower() in IMG_EXTENSIONS:
            sample_images.append({
                'path': os.path.join(cls_path, img_file),
                'class': cls_folder
            })

# Random sample of 16
random.seed(SEED)
batch_sample = random.sample(sample_images, min(16, len(sample_images)))

# Plot 4x4 grid
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for idx, item in enumerate(batch_sample):
    try:
        img = Image.open(item['path']).convert('RGB')
        axes[idx].imshow(img)
        axes[idx].set_title(item['class'], fontsize=10, fontweight='bold',
                           color='#0D6E6E')
    except Exception as e:
        axes[idx].text(0.5, 0.5, f"Error", ha='center', va='center',
                       transform=axes[idx].transAxes)
    axes[idx].axis('off')

# Hide extra axes if < 16 images
for idx in range(len(batch_sample), 16):
    axes[idx].axis('off')

fig.suptitle("Sample Training Batch (16 images)",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"\n✅ Displayed {len(batch_sample)} sample training images.")

In [ ]:
# ============================================================
# STEP 3.3 — Estimate training time and print GO confirmation
# ============================================================

# Rough time estimates based on hardware and dataset size
if GPU_AVAILABLE:
    if 'T4' in GPU_NAME:
        secs_per_epoch = max(5, train_total / (BATCH_SIZE * 15))
    elif 'V100' in GPU_NAME or 'A100' in GPU_NAME:
        secs_per_epoch = max(3, train_total / (BATCH_SIZE * 30))
    else:
        secs_per_epoch = max(5, train_total / (BATCH_SIZE * 10))
else:
    secs_per_epoch = max(30, train_total / (BATCH_SIZE * 2))

total_estimated_secs = secs_per_epoch * TRAIN_CONFIG['epochs']
est_minutes = total_estimated_secs / 60

print("=" * 60)
print("⏱️  TRAINING TIME ESTIMATE")
print("=" * 60)
print(f"  Dataset size    : {train_total} training images")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Epochs          : {TRAIN_CONFIG['epochs']}")
print(f"  Steps per epoch : ~{train_total // BATCH_SIZE}")
print(f"  Est. per epoch  : ~{secs_per_epoch:.0f} seconds")
print(f"  Est. total time : ~{est_minutes:.0f} minutes ({est_minutes/60:.1f} hours)")

if TRAIN_CONFIG['patience'] > 0:
    print(f"\n  📌 Early stopping is ON (patience={TRAIN_CONFIG['patience']})")
    print(f"     Training may finish earlier if validation loss plateaus.")

print("\n" + "=" * 60)
print("╔══════════════════════════════════════════════════╗")
print("║        🟢  GO FOR TRAINING — ALL CHECKS PASSED  ║")
print("╚══════════════════════════════════════════════════╝")
print("=" * 60)

# ═══════════════════════════════════════════
# STEP 4 — MODEL TRAINING
# ═══════════════════════════════════════════

**What this step does:**  
Loads the YOLOv8s-cls pretrained model and trains it on the BCNatal fetal ultrasound dataset.  
YOLOv8 automatically handles augmentation, logging, and checkpoint saving.

**Expected output:**  
Training progress with per-epoch metrics, then a summary with best.pt location and total time.

In [ ]:
# ============================================================
# STEP 4.1 — Load pretrained model
# ============================================================

print(f"📥 Loading pretrained model: {MODEL_VARIANT}")
model = YOLO(MODEL_VARIANT)
print(f"✅ Model loaded successfully!")
print(f"   Architecture : YOLOv8s Classification")
print(f"   Parameters   : ~6.4M (small variant)")
print(f"   Pretrained   : ImageNet")

In [ ]:
# ============================================================
# STEP 4.2 — Train the model
# ============================================================
# This is the main training cell. It uses all the config from
# Step 2. Training progress is printed per epoch.
#
# During training, YOLOv8 will automatically:
#   - Apply data augmentation (flip, rotate, color jitter, etc.)
#   - Log metrics per epoch (loss, accuracy, etc.)
#   - Save best.pt (best validation accuracy) and last.pt
#   - Generate training visualizations
#
# If training is interrupted, you can resume from the last
# checkpoint by changing MODEL_VARIANT to the last.pt path.

print("=" * 60)
print("🚀 STARTING TRAINING")
print("=" * 60)
print(f"   Model      : {MODEL_VARIANT}")
print(f"   Task       : {TASK_TYPE}")
print(f"   Epochs     : {TRAIN_CONFIG['epochs']}")
print(f"   Batch size : {TRAIN_CONFIG['batch']}")
print(f"   Image size : {TRAIN_CONFIG['imgsz']}")
print(f"   Device     : {'GPU (' + GPU_NAME + ')' if GPU_AVAILABLE else 'CPU'}")
print(f"   Save to    : {RUNS_DIR}/{RUN_NAME}")
print("=" * 60)
print()

# Record training start time
train_start_time = time.time()

# ---- Train ----
results = model.train(
    data=TRAIN_CONFIG['data'],
    epochs=TRAIN_CONFIG['epochs'],
    imgsz=TRAIN_CONFIG['imgsz'],
    batch=TRAIN_CONFIG['batch'],
    optimizer=TRAIN_CONFIG['optimizer'],
    lr0=TRAIN_CONFIG['lr0'],
    lrf=TRAIN_CONFIG['lrf'],
    patience=TRAIN_CONFIG['patience'],
    dropout=TRAIN_CONFIG['dropout'],
    seed=TRAIN_CONFIG['seed'],
    device=TRAIN_CONFIG['device'],
    project=TRAIN_CONFIG['project'],
    name=TRAIN_CONFIG['name'],
    save=TRAIN_CONFIG['save'],
    save_period=TRAIN_CONFIG['save_period'],
    val=TRAIN_CONFIG['val'],
    plots=TRAIN_CONFIG['plots'],
    exist_ok=TRAIN_CONFIG['exist_ok'],
    verbose=TRAIN_CONFIG['verbose'],
    workers=TRAIN_CONFIG['workers'],
    pretrained=TRAIN_CONFIG['pretrained'],
)

# Record training end time
train_end_time = time.time()
training_duration_secs = train_end_time - train_start_time
training_duration_mins = training_duration_secs / 60
training_duration_str = str(timedelta(seconds=int(training_duration_secs)))

print("\n" + "=" * 60)
print("🎉 TRAINING COMPLETE!")
print("=" * 60)
print(f"  Total training time : {training_duration_str} ({training_duration_mins:.1f} min)")

In [ ]:
# ============================================================
# STEP 4.3 — Post-training summary
# ============================================================
# Print the location of the best model and list all saved files.

# Determine the actual run directory
RUN_DIR = os.path.join(RUNS_DIR, RUN_NAME)

# If YOLO appended a number (e.g., fetal_yolov8s_v12), find it
if not os.path.exists(RUN_DIR):
    possible_runs = sorted(glob.glob(os.path.join(RUNS_DIR, f"{RUN_NAME}*")))
    if possible_runs:
        RUN_DIR = possible_runs[-1]  # Use latest

BEST_MODEL_PATH = os.path.join(RUN_DIR, 'weights', 'best.pt')
LAST_MODEL_PATH = os.path.join(RUN_DIR, 'weights', 'last.pt')

print("=" * 60)
print("📁 TRAINING OUTPUT FILES")
print("=" * 60)
print(f"  Run directory : {RUN_DIR}")
print(f"  Best model    : {BEST_MODEL_PATH}")
print(f"  Last model    : {LAST_MODEL_PATH}")
print()

# List all files in the run directory
if os.path.exists(RUN_DIR):
    print("  Files saved:")
    for root, dirs, files in os.walk(RUN_DIR):
        level = root.replace(RUN_DIR, '').count(os.sep)
        indent = '  ' + '│   ' * level
        subdir_name = os.path.basename(root)
        print(f"{indent}📁 {subdir_name}/")
        sub_indent = '  ' + '│   ' * (level + 1)
        for f in sorted(files):
            fsize = os.path.getsize(os.path.join(root, f))
            if fsize > 1024*1024:
                size_str = f"{fsize/(1024*1024):.1f} MB"
            elif fsize > 1024:
                size_str = f"{fsize/1024:.1f} KB"
            else:
                size_str = f"{fsize} B"
            print(f"{sub_indent}📄 {f} ({size_str})")
else:
    print("  ⚠️ Run directory not found!")

# Print best model size
if os.path.exists(BEST_MODEL_PATH):
    best_size_mb = os.path.getsize(BEST_MODEL_PATH) / (1024*1024)
    print(f"\n  ✅ Best model size: {best_size_mb:.1f} MB")

print("=" * 60)

# ═══════════════════════════════════════════
# STEP 5 — TRAINING RESULTS VISUALIZATION
# ═══════════════════════════════════════════

**What this step does:**  
Loads the training results CSV and plots accuracy, loss, precision, recall,  
and F1-score curves across epochs for both train and validation sets.

**Expected output:**  
A multi-panel figure with 5 subplots saved as `training_results.png` to Google Drive.

In [ ]:
# ============================================================
# STEP 5.1 — Load training results CSV
# ============================================================

results_csv_path = os.path.join(RUN_DIR, 'results.csv')

if os.path.exists(results_csv_path):
    results_df = pd.read_csv(results_csv_path)
    # Clean column names (YOLO sometimes adds extra spaces)
    results_df.columns = [c.strip() for c in results_df.columns]
    print(f"✅ Results loaded: {len(results_df)} epochs")
    print(f"   Columns: {list(results_df.columns)}")
    print()
    # Show last 5 rows
    print(results_df.tail(5).to_string(index=False))
else:
    print(f"⚠️ results.csv not found at: {results_csv_path}")
    print("   Check if training completed successfully.")

In [ ]:
# ============================================================
# STEP 5.2 — Plot training curves (5-panel figure)
# ============================================================
# Medical teal (#0D6E6E) for train curves
# Coral (#E05C5C) for validation curves

TEAL  = '#0D6E6E'
CORAL = '#E05C5C'
TEAL_LIGHT  = '#2EACAC'
CORAL_LIGHT = '#F09090'

# Identify available columns (YOLO naming varies by version)
# Classification results typically have:
#   train/loss, val/loss, metrics/accuracy_top1, metrics/accuracy_top5

def find_col(df, candidates):
    """Find the first matching column from a list of candidates."""
    for c in candidates:
        if c in df.columns:
            return c
        # Try partial match
        for col in df.columns:
            if c.lower() in col.lower():
                return col
    return None

# Map metric names to possible column names
col_epoch      = find_col(results_df, ['epoch', 'Epoch'])
col_train_loss = find_col(results_df, ['train/loss', 'train_loss'])
col_val_loss   = find_col(results_df, ['val/loss', 'val_loss'])
col_acc_top1   = find_col(results_df, ['metrics/accuracy_top1', 'accuracy_top1', 'top1_acc'])
col_acc_top5   = find_col(results_df, ['metrics/accuracy_top5', 'accuracy_top5', 'top5_acc'])

# For classification, YOLO may not directly output precision/recall/F1 per epoch.
# We extract what's available and handle missing gracefully.
col_precision = find_col(results_df, ['metrics/precision', 'precision', 'Precision'])
col_recall    = find_col(results_df, ['metrics/recall', 'recall', 'Recall'])
col_f1        = find_col(results_df, ['metrics/f1', 'f1', 'F1'])

print("Detected columns:")
for name, col in [('Epoch', col_epoch), ('Train Loss', col_train_loss),
                  ('Val Loss', col_val_loss), ('Accuracy Top-1', col_acc_top1),
                  ('Accuracy Top-5', col_acc_top5), ('Precision', col_precision),
                  ('Recall', col_recall), ('F1', col_f1)]:
    status = col if col else '⚠️ NOT FOUND'
    print(f"  {name:<18}: {status}")

In [ ]:
# ============================================================
# STEP 5.3 — Generate the 5-panel training visualization
# ============================================================

epochs = results_df[col_epoch] if col_epoch else results_df.index + 1

# Determine how many subplots we need based on available data
plot_configs = []

# Panel 1: Top-1 Accuracy
if col_acc_top1:
    plot_configs.append({
        'title': 'Top-1 Accuracy',
        'ylabel': 'Accuracy',
        'lines': [{'data': results_df[col_acc_top1], 'label': 'Accuracy (Top-1)',
                   'color': TEAL, 'ls': '-'}]
    })
    # Add Top-5 if available
    if col_acc_top5:
        plot_configs[-1]['lines'].append({
            'data': results_df[col_acc_top5], 'label': 'Accuracy (Top-5)',
            'color': TEAL_LIGHT, 'ls': '--'
        })

# Panel 2: Loss
loss_lines = []
if col_train_loss:
    loss_lines.append({'data': results_df[col_train_loss], 'label': 'Train Loss',
                       'color': TEAL, 'ls': '-'})
if col_val_loss:
    loss_lines.append({'data': results_df[col_val_loss], 'label': 'Val Loss',
                       'color': CORAL, 'ls': '-'})
if loss_lines:
    plot_configs.append({'title': 'Loss', 'ylabel': 'Loss', 'lines': loss_lines})

# Panel 3: Precision (if available, else use accuracy again as placeholder)
if col_precision:
    plot_configs.append({
        'title': 'Precision', 'ylabel': 'Precision',
        'lines': [{'data': results_df[col_precision], 'label': 'Precision',
                   'color': TEAL, 'ls': '-'}]
    })

# Panel 4: Recall
if col_recall:
    plot_configs.append({
        'title': 'Recall', 'ylabel': 'Recall',
        'lines': [{'data': results_df[col_recall], 'label': 'Recall',
                   'color': TEAL, 'ls': '-'}]
    })

# Panel 5: F1 Score
if col_f1:
    plot_configs.append({
        'title': 'F1 Score', 'ylabel': 'F1',
        'lines': [{'data': results_df[col_f1], 'label': 'F1 Score',
                   'color': TEAL, 'ls': '-'}]
    })

# If we have fewer than 5 panels, add learning rate curve
col_lr = find_col(results_df, ['lr/pg0', 'lr0', 'learning_rate'])
if col_lr and len(plot_configs) < 5:
    plot_configs.append({
        'title': 'Learning Rate', 'ylabel': 'LR',
        'lines': [{'data': results_df[col_lr], 'label': 'Learning Rate',
                   'color': '#555555', 'ls': '-'}]
    })

# Ensure at least 2 panels
n_panels = max(len(plot_configs), 2)

# Create figure
fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
if n_panels == 1:
    axes = [axes]

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    for line in config['lines']:
        ax.plot(epochs, line['data'], color=line['color'],
                linestyle=line.get('ls', '-'), linewidth=2,
                label=line['label'], marker='', markersize=3)

    ax.set_title(config['title'], fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(config['ylabel'], fontsize=11)
    ax.legend(fontsize=9, loc='best')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Hide extra axes
for idx in range(len(plot_configs), len(axes)):
    axes[idx].axis('off')

fig.suptitle('YOLOv8s Classification — Training Results',
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()

# Save to Google Drive
training_results_path = os.path.join(DRIVE_DIR, 'training_results.png')
fig.savefig(training_results_path, dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()

print(f"\n✅ Training results figure saved to: {training_results_path}")

# ═══════════════════════════════════════════
# STEP 6 — MODEL EVALUATION ON TEST SET
# ═══════════════════════════════════════════

**What this step does:**  
Loads the best trained model and evaluates it on the held-out test set.  
Computes accuracy, precision, recall, F1, Top-5 accuracy, and confusion matrix.

**Expected output:**  
A per-class metrics table, confusion matrix heatmap, and saved reports.

In [ ]:
# ============================================================
# STEP 6.1 — Load the best model
# ============================================================

print("=" * 60)
print("📥 LOADING BEST MODEL FOR EVALUATION")
print("=" * 60)

if os.path.exists(BEST_MODEL_PATH):
    best_model = YOLO(BEST_MODEL_PATH)
    best_size_mb = os.path.getsize(BEST_MODEL_PATH) / (1024*1024)
    print(f"  ✅ Loaded: {BEST_MODEL_PATH}")
    print(f"  Size: {best_size_mb:.1f} MB")
else:
    print(f"  ⚠️ best.pt not found at: {BEST_MODEL_PATH}")
    print("  Trying last.pt instead...")
    best_model = YOLO(LAST_MODEL_PATH)
    print(f"  ✅ Loaded: {LAST_MODEL_PATH}")

print("=" * 60)

In [ ]:
# ============================================================
# STEP 6.2 — Run inference on the entire test set
# ============================================================
# We predict on every test image and collect true vs predicted
# labels for computing metrics.

print("🧪 Running inference on the test set...")

test_dir = os.path.join(DATASET_DIR, 'test')

# Collect all test images with their true labels
all_true_labels  = []
all_pred_labels  = []
all_pred_confs   = []
all_pred_probs   = []  # Full probability vectors for Top-5
all_image_paths  = []

# Build class name → index mapping
# Use the order from the training data.yaml
test_classes = sorted(os.listdir(test_dir))
test_classes = [c for c in test_classes if os.path.isdir(os.path.join(test_dir, c))]

# Get the model's own class names (from training)
model_class_names = best_model.names  # dict: {idx: name}
print(f"  Model classes: {model_class_names}")

# Iterate through test images
total_test_images = 0
for cls_folder in test_classes:
    cls_path = os.path.join(test_dir, cls_folder)
    for img_file in sorted(os.listdir(cls_path)):
        if os.path.splitext(img_file)[1].lower() in IMG_EXTENSIONS:
            total_test_images += 1

print(f"  Total test images to evaluate: {total_test_images}")
print()

processed = 0
for cls_folder in test_classes:
    cls_path = os.path.join(test_dir, cls_folder)
    img_files = [f for f in sorted(os.listdir(cls_path))
                 if os.path.splitext(f)[1].lower() in IMG_EXTENSIONS]

    # Batch prediction for efficiency
    batch_paths = [os.path.join(cls_path, f) for f in img_files]

    if not batch_paths:
        continue

    # Predict in batches
    batch_results = best_model.predict(
        source=batch_paths,
        imgsz=TRAIN_CONFIG['imgsz'],
        device=DEVICE,
        verbose=False
    )

    for img_path, result in zip(batch_paths, batch_results):
        probs = result.probs
        pred_idx = probs.top1
        pred_conf = probs.top1conf.item()
        pred_name = model_class_names[pred_idx]

        all_true_labels.append(cls_folder)
        all_pred_labels.append(pred_name)
        all_pred_confs.append(pred_conf)
        all_pred_probs.append(probs.data.cpu().numpy())
        all_image_paths.append(img_path)

        processed += 1

    print(f"  Processed: {processed}/{total_test_images} ({cls_folder})")

print(f"\n✅ Inference complete on {len(all_true_labels)} test images.")

In [ ]:
# ============================================================
# STEP 6.3 — Compute overall metrics
# ============================================================

# Overall accuracy
overall_accuracy = accuracy_score(all_true_labels, all_pred_labels)

# Top-5 accuracy
# Build label encoder mapping
unique_labels = sorted(set(all_true_labels + all_pred_labels))
label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}

# Convert true labels to indices for top-k
true_indices = np.array([label_to_idx.get(l, 0) for l in all_true_labels])

# Build probability matrix aligned with unique_labels ordering
prob_matrix = np.zeros((len(all_pred_probs), len(unique_labels)))
for i, probs_vec in enumerate(all_pred_probs):
    for model_idx, model_name in model_class_names.items():
        if model_name in label_to_idx:
            col_idx = label_to_idx[model_name]
            if model_idx < len(probs_vec):
                prob_matrix[i, col_idx] = probs_vec[model_idx]

# Compute Top-5 accuracy (k = min(5, num_classes))
k_val = min(5, len(unique_labels))
try:
    top5_accuracy = top_k_accuracy_score(
        true_indices, prob_matrix, k=k_val, labels=list(range(len(unique_labels)))
    )
except Exception:
    top5_accuracy = -1.0  # Fallback

# Per-class + macro metrics
precision_macro = precision_score(all_true_labels, all_pred_labels, average='macro', zero_division=0)
recall_macro    = recall_score(all_true_labels, all_pred_labels, average='macro', zero_division=0)
f1_macro        = f1_score(all_true_labels, all_pred_labels, average='macro', zero_division=0)

# Classification report
report = classification_report(
    all_true_labels, all_pred_labels,
    target_names=unique_labels,
    output_dict=True,
    zero_division=0
)

# ---- Print overall summary ----
print("\n" + "=" * 60)
print("📊 OVERALL PERFORMANCE METRICS")
print("=" * 60)
print(f"  Overall Accuracy  : {overall_accuracy*100:.2f}%")
if top5_accuracy >= 0:
    print(f"  Top-5 Accuracy    : {top5_accuracy*100:.2f}%")
print(f"  Macro Precision   : {precision_macro*100:.2f}%")
print(f"  Macro Recall      : {recall_macro*100:.2f}%")
print(f"  Macro F1 Score    : {f1_macro*100:.2f}%")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 6.4 — Per-class performance table
# ============================================================

print("\n" + "=" * 75)
print("📋 PER-CLASS PERFORMANCE")
print("=" * 75)
print(f"  {'Class':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print(f"  {'─'*25} {'─'*10} {'─'*10} {'─'*10} {'─'*10}")

metrics_rows = []

for cls in unique_labels:
    if cls in report:
        p = report[cls]['precision']
        r = report[cls]['recall']
        f = report[cls]['f1-score']
        s = int(report[cls]['support'])
        print(f"  {cls:<25} {p*100:>9.1f}% {r*100:>9.1f}% {f*100:>9.1f}% {s:>10}")
        metrics_rows.append({
            'Class': cls, 'Precision': round(p, 4),
            'Recall': round(r, 4), 'F1': round(f, 4), 'Support': s
        })

print(f"  {'─'*25} {'─'*10} {'─'*10} {'─'*10} {'─'*10}")
print(f"  {'MACRO AVERAGE':<25} {precision_macro*100:>9.1f}% {recall_macro*100:>9.1f}% {f1_macro*100:>9.1f}% {len(all_true_labels):>10}")

# Add macro row
metrics_rows.append({
    'Class': 'MACRO AVERAGE',
    'Precision': round(precision_macro, 4),
    'Recall': round(recall_macro, 4),
    'F1': round(f1_macro, 4),
    'Support': len(all_true_labels)
})

print("=" * 75)

# Save metrics to CSV
metrics_df = pd.DataFrame(metrics_rows)
metrics_csv_path = os.path.join(DRIVE_DIR, 'metrics_report.csv')
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"\n✅ Metrics saved to: {metrics_csv_path}")

In [ ]:
# ============================================================
# STEP 6.5 — Confusion matrix heatmap
# ============================================================

cm = confusion_matrix(all_true_labels, all_pred_labels, labels=unique_labels)

# Normalize confusion matrix (percentage per true class)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_normalized = np.nan_to_num(cm_normalized)  # Handle division by zero

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

# ---- Left: Raw counts ----
sns.heatmap(
    cm, annot=True, fmt='d', cmap='YlGnBu',
    xticklabels=unique_labels, yticklabels=unique_labels,
    ax=axes[0], linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Count'}
)
axes[0].set_xlabel('Predicted Class', fontsize=12)
axes[0].set_ylabel('True Class', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=9)
axes[0].tick_params(axis='y', rotation=0, labelsize=9)

# ---- Right: Normalized (percentage) ----
# Build a custom teal-based colormap
from matplotlib.colors import LinearSegmentedColormap
teal_cmap = LinearSegmentedColormap.from_list(
    'teal', ['#FFFFFF', '#B2DFDB', '#4DB6AC', '#0D6E6E', '#004D40']
)

sns.heatmap(
    cm_normalized, annot=True, fmt='.1%', cmap=teal_cmap,
    xticklabels=unique_labels, yticklabels=unique_labels,
    ax=axes[1], linewidths=0.5, linecolor='white',
    vmin=0, vmax=1, cbar_kws={'label': 'Proportion'}
)
axes[1].set_xlabel('Predicted Class', fontsize=12)
axes[1].set_ylabel('True Class', fontsize=12)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=9)
axes[1].tick_params(axis='y', rotation=0, labelsize=9)

fig.suptitle(f'Test Set Evaluation — Overall Accuracy: {overall_accuracy*100:.1f}%',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

# Save
cm_path = os.path.join(DRIVE_DIR, 'confusion_matrix.png')
fig.savefig(cm_path, dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()

print(f"\n✅ Confusion matrix saved to: {cm_path}")

# ═══════════════════════════════════════════
# STEP 7 — SAMPLE INFERENCE & VISUALIZATION
# ═══════════════════════════════════════════

**What this step does:**  
Runs inference on 9 test images (one per class) and displays them in a 3×3 grid  
with predicted label, confidence score, true label, and correct/wrong indicator.

**Expected output:**  
A 3×3 grid image saved as `sample_predictions.png` to Google Drive.

In [ ]:
# ============================================================
# STEP 7 — Sample inference on 9 test images (1 per class)
# ============================================================

print("=" * 60)
print("🔍 SAMPLE INFERENCE — 1 image per class")
print("=" * 60)

# Select 1 random image per class from the test set
random.seed(SEED)
sample_items = []

for cls_folder in sorted(test_classes):
    cls_path = os.path.join(test_dir, cls_folder)
    img_files = [f for f in os.listdir(cls_path)
                 if os.path.splitext(f)[1].lower() in IMG_EXTENSIONS]
    if img_files:
        chosen = random.choice(img_files)
        sample_items.append({
            'path': os.path.join(cls_path, chosen),
            'true_class': cls_folder
        })

# Limit to 9 for 3x3 grid
sample_items = sample_items[:9]

# Run prediction on samples
sample_paths = [item['path'] for item in sample_items]
sample_results = best_model.predict(
    source=sample_paths,
    imgsz=TRAIN_CONFIG['imgsz'],
    device=DEVICE,
    verbose=False
)

# ---- Plot 3x3 grid ----
n_samples = len(sample_items)
n_cols = 3
n_rows = (n_samples + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for idx, (item, result) in enumerate(zip(sample_items, sample_results)):
    img = Image.open(item['path']).convert('RGB')

    probs = result.probs
    pred_idx = probs.top1
    pred_conf = probs.top1conf.item() * 100  # percentage
    pred_name = model_class_names[pred_idx]
    true_name = item['true_class']

    is_correct = pred_name == true_name
    indicator = "✓" if is_correct else "✗"
    border_color = '#0D6E6E' if is_correct else '#E05C5C'

    axes[idx].imshow(img)

    title_text = f"Pred: {pred_name} ({pred_conf:.1f}%)\nTrue: {true_name}  {indicator}"
    title_color = '#0D6E6E' if is_correct else '#E05C5C'
    axes[idx].set_title(title_text, fontsize=11, fontweight='bold',
                        color=title_color)

    # Add colored border
    for spine in axes[idx].spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
    axes[idx].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Hide extra subplots
for idx in range(n_samples, len(axes)):
    axes[idx].axis('off')

correct_count = sum(1 for item, res in zip(sample_items, sample_results)
                    if model_class_names[res.probs.top1] == item['true_class'])

fig.suptitle(
    f'Sample Predictions — {correct_count}/{n_samples} Correct',
    fontsize=16, fontweight='bold', y=1.02
)
plt.tight_layout()

# Save
predictions_path = os.path.join(DRIVE_DIR, 'sample_predictions.png')
fig.savefig(predictions_path, dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()

print(f"\n✅ Sample predictions saved to: {predictions_path}")

# ═══════════════════════════════════════════
# STEP 8 — MODEL EXPORT
# ═══════════════════════════════════════════

**What this step does:**  
Exports the best trained model to TorchScript and ONNX formats for deployment.  
Prints file sizes and recommends the best format for the web demo.

**Expected output:**  
Exported model files with sizes, plus a deployment recommendation.

In [ ]:
# ============================================================
# STEP 8.1 — Export to TorchScript
# ============================================================

print("=" * 60)
print("📦 MODEL EXPORT")
print("=" * 60)

export_results = {}

# ---- TorchScript ----
print("\n🔧 Exporting to TorchScript...")
try:
    ts_path = best_model.export(
        format='torchscript',
        imgsz=TRAIN_CONFIG['imgsz'],
        device=DEVICE
    )
    if os.path.exists(ts_path):
        ts_size_mb = os.path.getsize(ts_path) / (1024*1024)
        export_results['TorchScript'] = {
            'path': ts_path, 'size_mb': ts_size_mb, 'status': '✅'
        }
        # Copy to Drive
        shutil.copy2(ts_path, os.path.join(DRIVE_DIR, os.path.basename(ts_path)))
        print(f"   ✅ Exported: {ts_path} ({ts_size_mb:.1f} MB)")
    else:
        export_results['TorchScript'] = {'path': '', 'size_mb': 0, 'status': '❌'}
        print(f"   ❌ Export failed")
except Exception as e:
    export_results['TorchScript'] = {'path': '', 'size_mb': 0, 'status': '❌'}
    print(f"   ❌ TorchScript export error: {e}")

In [ ]:
# ============================================================
# STEP 8.2 — Export to ONNX
# ============================================================

print("🔧 Exporting to ONNX...")
try:
    onnx_path = best_model.export(
        format='onnx',
        imgsz=TRAIN_CONFIG['imgsz'],
        simplify=True,
        dynamic=False
    )
    if os.path.exists(onnx_path):
        onnx_size_mb = os.path.getsize(onnx_path) / (1024*1024)
        export_results['ONNX'] = {
            'path': onnx_path, 'size_mb': onnx_size_mb, 'status': '✅'
        }
        # Copy to Drive
        shutil.copy2(onnx_path, os.path.join(DRIVE_DIR, os.path.basename(onnx_path)))
        print(f"   ✅ Exported: {onnx_path} ({onnx_size_mb:.1f} MB)")
    else:
        export_results['ONNX'] = {'path': '', 'size_mb': 0, 'status': '❌'}
        print(f"   ❌ Export failed")
except Exception as e:
    export_results['ONNX'] = {'path': '', 'size_mb': 0, 'status': '❌'}
    print(f"   ❌ ONNX export error: {e}")

In [ ]:
# ============================================================
# STEP 8.3 — Export to TFLite (optional, for mobile/edge)
# ============================================================

print("🔧 Exporting to TFLite (optional)...")
try:
    tflite_path = best_model.export(
        format='tflite',
        imgsz=TRAIN_CONFIG['imgsz']
    )
    if os.path.exists(tflite_path):
        tflite_size_mb = os.path.getsize(tflite_path) / (1024*1024)
        export_results['TFLite'] = {
            'path': tflite_path, 'size_mb': tflite_size_mb, 'status': '✅'
        }
        shutil.copy2(tflite_path, os.path.join(DRIVE_DIR, os.path.basename(tflite_path)))
        print(f"   ✅ Exported: {tflite_path} ({tflite_size_mb:.1f} MB)")
    else:
        export_results['TFLite'] = {'path': '', 'size_mb': 0, 'status': '⚠️ N/A'}
        print(f"   ⚠️ Export produced no file (optional — not critical)")
except Exception as e:
    export_results['TFLite'] = {'path': '', 'size_mb': 0, 'status': '⚠️ Skipped'}
    print(f"   ⚠️ TFLite export skipped: {e}")
    print(f"   (This is optional and not needed for web deployment)")

In [ ]:
# ============================================================
# STEP 8.4 — Export summary and deployment recommendation
# ============================================================

# Also copy best.pt to Drive root for easy access
if os.path.exists(BEST_MODEL_PATH):
    shutil.copy2(BEST_MODEL_PATH, os.path.join(DRIVE_DIR, 'best.pt'))

print("\n" + "=" * 60)
print("📦 EXPORT SUMMARY")
print("=" * 60)
print(f"  {'Format':<14} {'Status':>8} {'Size':>10} {'Path'}")
print(f"  {'─'*14} {'─'*8} {'─'*10} {'─'*30}")
print(f"  {'PyTorch (.pt)':<14} {'✅':>8} {best_size_mb:>9.1f}M {BEST_MODEL_PATH}")

for fmt, info in export_results.items():
    status = info['status']
    size = f"{info['size_mb']:.1f}M" if info['size_mb'] > 0 else 'N/A'
    path = info['path'] if info['path'] else 'N/A'
    print(f"  {fmt:<14} {status:>8} {size:>10} {path}")

print("\n" + "-" * 60)
print("💡 DEPLOYMENT RECOMMENDATION")
print("-" * 60)
print("  For Antigravity web demo (Phase 4):")
print("")
print("  ▸ PRIMARY: Use ONNX format (.onnx)")
print("    Reason: Cross-platform, can run in browser via ONNX Runtime Web,")
print("    or server-side with Python/Node.js. Best balance of speed + portability.")
print("")
print("  ▸ ALTERNATIVE: Use PyTorch (.pt) with a Flask/FastAPI backend")
print("    Reason: Easiest to set up, full Ultralytics API support,")
print("    but requires a Python server.")
print("")
print("  ▸ MOBILE: Use TFLite (.tflite) if deploying to Android/iOS")
print("=" * 60)

# ═══════════════════════════════════════════
# STEP 9 — MODEL CARD & SUMMARY
# ═══════════════════════════════════════════

**What this step does:**  
Generates a comprehensive `model_card.md` file documenting the model's architecture,  
training configuration, performance metrics, and usage instructions.

**Expected output:**  
A Markdown-formatted model card saved to Google Drive.

In [ ]:
# ============================================================
# STEP 9 — Generate model_card.md
# ============================================================

# Gather actual values from training
actual_epochs = len(results_df) if 'results_df' in dir() else TRAIN_CONFIG['epochs']

# Find best epoch (highest top-1 accuracy)
best_epoch = 0
best_acc_val = overall_accuracy
if col_acc_top1 and 'results_df' in dir():
    best_epoch = int(results_df[col_acc_top1].idxmax()) + 1
    best_acc_val = results_df[col_acc_top1].max()

today_str = datetime.now().strftime('%Y-%m-%d')

model_card_content = f"""# Model Card — Fetal Image Analysis (YOLOv8s Classification)

---

## Project Information

| Field | Value |
|-------|-------|
| **Project** | Fetal Image Analysis Using Deep Learning |
| **Organization** | TB Solutions |
| **Type** | ECE College Microproject |
| **Date** | {today_str} |
| **Trained by** | TB Solutions |

---

## Model Details

| Field | Value |
|-------|-------|
| **Architecture** | YOLOv8s Classification |
| **Framework** | Ultralytics {ultralytics.__version__} + PyTorch {torch.__version__} |
| **Task** | Multi-class Image Classification |
| **Input Size** | {TRAIN_CONFIG['imgsz']}×{TRAIN_CONFIG['imgsz']} pixels |
| **Parameters** | ~6.4M |
| **Model Size** | {best_size_mb:.1f} MB (.pt) |
| **Pretrained On** | ImageNet |

---

## Dataset

| Field | Value |
|-------|-------|
| **Dataset** | BCNatal Maternal-Fetal Ultrasound Dataset |
| **Source** | [Zenodo Record 3904280](https://zenodo.org/record/3904280) |
| **Total Images** | {grand_total} |
| **Training Images** | {train_total} (70%) |
| **Validation Images** | {val_total} (20%) |
| **Test Images** | {test_total} (10%) |
| **Number of Classes** | {NUM_CLASSES} |
| **Classes** | {', '.join(CLASS_NAMES)} |

---

## Training Configuration

| Parameter | Value |
|-----------|-------|
| **Epochs** | {actual_epochs} (trained) / {TRAIN_CONFIG['epochs']} (configured) |
| **Best Epoch** | {best_epoch} |
| **Batch Size** | {TRAIN_CONFIG['batch']} |
| **Optimizer** | {TRAIN_CONFIG['optimizer']} |
| **Learning Rate** | {TRAIN_CONFIG['lr0']} (initial), {TRAIN_CONFIG['lrf']} (final factor) |
| **Dropout** | {TRAIN_CONFIG['dropout']} |
| **Early Stopping** | Patience = {TRAIN_CONFIG['patience']} |
| **Random Seed** | {SEED} |
| **Training Time** | {training_duration_str} |
| **Hardware** | {GPU_NAME} ({GPU_VRAM_GB:.1f} GB VRAM) |

---

## Performance Metrics (Test Set)

| Metric | Value |
|--------|-------|
| **Overall Accuracy** | {overall_accuracy*100:.2f}% |
| **Top-5 Accuracy** | {top5_accuracy*100:.2f}% |
| **Macro Precision** | {precision_macro*100:.2f}% |
| **Macro Recall** | {recall_macro*100:.2f}% |
| **Macro F1 Score** | {f1_macro*100:.2f}% |

### Per-Class Performance

| Class | Precision | Recall | F1 Score | Support |
|-------|-----------|--------|----------|---------|
"""

# Add per-class rows
for row in metrics_rows:
    if row['Class'] != 'MACRO AVERAGE':
        model_card_content += f"| {row['Class']} | {row['Precision']*100:.1f}% | {row['Recall']*100:.1f}% | {row['F1']*100:.1f}% | {row['Support']} |\n"
model_card_content += f"| **MACRO AVG** | **{precision_macro*100:.1f}%** | **{recall_macro*100:.1f}%** | **{f1_macro*100:.1f}%** | **{len(all_true_labels)}** |\n"

model_card_content += f"""
---

## Export Formats

| Format | Size | Recommended Use |
|--------|------|----------------|
| PyTorch (.pt) | {best_size_mb:.1f} MB | Python inference, further training |
"""

for fmt, info in export_results.items():
    if info['size_mb'] > 0:
        use_case = {'TorchScript': 'Python deployment without Ultralytics',
                    'ONNX': 'Cross-platform, web deployment (recommended)',
                    'TFLite': 'Mobile / edge deployment'}.get(fmt, 'General')
        model_card_content += f"| {fmt} | {info['size_mb']:.1f} MB | {use_case} |\n"

model_card_content += f"""
---

## Usage

```python
from ultralytics import YOLO

# Load model
model = YOLO('best.pt')

# Predict on a fetal ultrasound image
results = model.predict('ultrasound_image.png', imgsz={TRAIN_CONFIG['imgsz']})

# Get predicted class and confidence
probs = results[0].probs
predicted_class = model.names[probs.top1]
confidence = probs.top1conf.item()
print(f"Predicted: {{predicted_class}} ({{confidence*100:.1f}}%)")
```

---

## Limitations & Ethical Considerations

- This model is for **educational/research purposes only** and should NOT be used for clinical diagnosis.
- Performance may vary on ultrasound images from different machines or populations.
- The model was trained on the BCNatal dataset which may not represent all fetal imaging scenarios.
- Always consult qualified medical professionals for clinical decisions.

---

*Generated automatically by the Phase 2 training pipeline.*
"""

# Save model card
model_card_path = os.path.join(DRIVE_DIR, 'model_card.md')
with open(model_card_path, 'w', encoding='utf-8') as f:
    f.write(model_card_content)

print(f"✅ Model card saved to: {model_card_path}")
print()
print("=" * 60)
print("📄 MODEL CARD PREVIEW")
print("=" * 60)
# Print first 50 lines
for i, line in enumerate(model_card_content.split('\n')[:50]):
    print(line)
print("\n... (see full file on Google Drive)")
print("=" * 60)

# ═══════════════════════════════════════════
# STEP 10 — FINAL CHECKLIST
# ═══════════════════════════════════════════

**What this step does:**  
Prints the final completion checklist and a comprehensive summary of everything  
accomplished in Phase 2, confirming readiness for Phase 3.

**Expected output:**  
A formatted checklist with all items marked ✓, plus a summary box.

In [ ]:
# ============================================================
# STEP 10 — Final checklist and summary
# ============================================================

# List all files saved to Drive
print("=" * 65)
print("📁 FILES SAVED TO GOOGLE DRIVE")
print("=" * 65)

if os.path.exists(DRIVE_DIR):
    for item in sorted(os.listdir(DRIVE_DIR)):
        item_path = os.path.join(DRIVE_DIR, item)
        if os.path.isfile(item_path):
            size = os.path.getsize(item_path)
            if size > 1024*1024:
                size_str = f"{size/(1024*1024):.1f} MB"
            elif size > 1024:
                size_str = f"{size/1024:.1f} KB"
            else:
                size_str = f"{size} B"
            print(f"  📄 {item:<35} {size_str:>10}")
        elif os.path.isdir(item_path):
            print(f"  📁 {item}/")

print("=" * 65)

In [ ]:
# ============================================================
# FINAL SUMMARY AND CHECKLIST
# ============================================================

print()
print("╔" + "═" * 65 + "╗")
print("║" + " 🩺  PHASE 2: MODEL TRAINING — COMPLETE!".center(65) + "║")
print("╠" + "═" * 65 + "╣")
print("║" + "".center(65) + "║")

summary_lines = [
    f"  Model             : YOLOv8s Classification",
    f"  Dataset           : BCNatal ({grand_total} images, {NUM_CLASSES} classes)",
    f"  Training Epochs   : {actual_epochs}",
    f"  Best Epoch        : {best_epoch}",
    f"  Training Time     : {training_duration_str}",
    f"  Overall Accuracy  : {overall_accuracy*100:.2f}%",
    f"  Macro F1 Score    : {f1_macro*100:.2f}%",
    f"  Model Size        : {best_size_mb:.1f} MB",
    f"  Export Formats    : PyTorch, TorchScript, ONNX",
    f"  GPU Used          : {GPU_NAME}",
    f"  Drive Location    : {DRIVE_DIR}",
]

for line in summary_lines:
    padded = f"║  {line:<63}║"
    print(padded)

print("║" + "".center(65) + "║")
print("╠" + "═" * 65 + "╣")
print("║" + " CHECKLIST".center(65) + "║")
print("╠" + "═" * 65 + "╣")

checklist = [
    f"[✓] GPU verified and training environment ready",
    f"[✓] YOLOv8s-cls model selected and configured",
    f"[✓] Pre-training checks passed",
    f"[✓] Model trained for {actual_epochs} epochs",
    f"[✓] Training results visualized and saved",
    f"[✓] Test set evaluation complete — Accuracy: {overall_accuracy*100:.1f}%",
    f"[✓] Sample inference grid saved",
    f"[✓] Model exported to TorchScript and ONNX",
    f"[✓] Model card saved",
    f"[✓] Ready for Phase 3 — Inference Pipeline",
]

for item in checklist:
    print(f"║  {item:<63}║")

print("║" + "".center(65) + "║")
print("╚" + "═" * 65 + "╝")
print()
print("🚀 READY FOR PHASE 3 — INFERENCE PIPELINE!")
print("   Next: Build the inference script for single image + batch prediction.")
print(f"   Model location: {DRIVE_DIR}/best.pt")